In [1]:

# 1) Environment Setup (Online-only flow)
deps = [
    "transformers>=4.44.0",
    "datasets>=2.18.0",
    "accelerate>=0.33.0",
    "peft>=0.12.0",
    "torch",
    "sacrebleu>=2.4.2",
    "evaluate>=0.4.2",
    "gradio>=4.44.0",
    "bitsandbytes>=0.43.1"
]

import sys, subprocess
for pkg in deps:
    try:
        __import__(pkg.split("==")[0].split(">=")[0].split("[")[0])
    except Exception:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ Environment ready")


Installing sacrebleu>=2.4.2 ...
Installing evaluate>=0.4.2 ...
Installing bitsandbytes>=0.43.1 ...
✅ Environment ready


In [ ]:
!nvidia-smi


Sat Nov  8 18:55:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2) Config - Optimized for speed
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # Smaller model (2x faster)
PROJECT_NAME = "gretel_text2sql_lora_online"
OUTPUT_DIR = "./outputs"
SEED = 42

# LoRA settings (speed optimized)
LORA_R = 8  # Reduced from 32 (4x less computation)
LORA_ALPHA = 16  # Proportional reduction
LORA_DROPOUT = 0.05
LR = 5e-5  # Keep conservative
EPOCHS = 1  # Single epoch instead of 3 (3x faster!)
WARMUP_RATIO = 0.03
GRADIENT_ACCUMULATION = 1  # Back to 1
MAX_LENGTH = 1024  # Reduced from 4096 (4x faster!)
LOGGING_STEPS = 50

# Generation settings
MAX_NEW_TOKENS = 128  # Reduced from 512 (2x faster)
MIN_NEW_TOKENS = 32
REPETITION_PENALTY = 1.05
BEAM_SIZE = 1  # Greedy instead of beam (2x faster!)

print("✅ Config loaded")

✅ Config loaded


## 3) Helpers — Prompt format, normalization, parsing

In [18]:
from typing import Dict, Any, List, Tuple
import json, os, random, re
import math
from dataclasses import dataclass

random.seed(SEED)

# Get the correct EOS token once tokenizer exists
def get_eos() -> str:
    try:
        eos = getattr(tokenizer, "eos_token", None)
    except NameError:
        eos = None
    return eos or "</s>"  # fallback only if tokenizer not ready

def build_prompt(schema: str, question: str) -> str:
    return (
        "<s>[SCHEMA]\n"
        + schema.strip()
        + "\n\n[QUESTION]\n"
        + question.strip()
        + "\n\n[RESPONSE_FORMAT]\n"
        + "SQL: "
    )

def build_target(sql: str, explanation: str = None) -> str:
    expl = (explanation or "This SQL answers the question using the provided schema.").strip()
    eos = get_eos()
    # Teach the model: SQL then Explanation then EOS
    return f"SQL: {sql.strip()}\nExplanation: {expl}{eos}"

_SQL_WS = re.compile(r"\s+")

def normalize_sql(s: str) -> str:
    s = s.strip()
    s = _SQL_WS.sub(" ", s)
    return s.lower()

# def split_response(model_text: str) -> Tuple[str, str]:
#     """
#     Parse model output into (sql, explanation).

#     Handles:
#     - Fine-tuned format:
#         SQL: ...
#         Explanation: ...
#     - Base-model style:
#         SELECT ...;
#         [EXPLANATION]
#         ...
#         [ANSWER]/[NOTES] ...
#     """
#     text = (model_text or "").strip()

#     # Cut at EOS if present
#     eos = get_eos()
#     if eos in text:
#         text = text.split(eos)[0].strip()

#     sql = ""
#     expl = ""

#     # 1) Preferred: labeled SQL / Explanation (fine-tuned)
#     sql_match = re.search(
#         r"(?:^|\n)\s*SQL\s*:\s*(.*?)(?:\n(?:Explanation|\[EXPLANATION\])\s*:|$)",
#         text,
#         flags=re.DOTALL | re.IGNORECASE,
#     )
#     exp_match = re.search(
#         r"(?:Explanation|\[EXPLANATION\])\s*:\s*(.*?)(?:\n\[[A-Z ]+\]|$)",
#         text,
#         flags=re.DOTALL | re.IGNORECASE,
#     )

#     if sql_match:
#         sql = sql_match.group(1).strip()
#     if exp_match:
#         expl = exp_match.group(1).strip()

#     # 2) Fallback: base model with [EXPLANATION] & no SQL label
#     if not sql:
#         sel_match = re.search(r"(?is)\bselect\b.*?;", text)
#         if sel_match:
#             sql = sel_match.group(0).strip()

#     if not expl:
#         ex_block = re.search(
#             r"\[EXPLANATION\]\s*(.*?)(?:\n\[[A-Z ]+\]|$)",
#             text,
#             flags=re.DOTALL,
#         )
#         if ex_block:
#             expl = ex_block.group(1).strip()

#     # 3) Fallback if still nothing
#     if not expl and sql:
#         expl = "(No explanation parsed.)"

#     return sql, expl
def split_response(model_text: str) -> Tuple[str, str]:
    """
    Parse model output into (sql, explanation).

    Handles:
    - Fine-tuned format:
        SQL: ...
        Explanation: ...
    - Base-model style (Qwen-ish):
        SQL: ...
        [ANSWER]
        ...
        [EXPLANATION]
        ...
    - Or just:
        SELECT ...;
        [EXPLANATION] ...
    """
    text = (model_text or "").strip()

    # Cut at EOS if present
    eos = get_eos()
    if eos in text:
        text = text.split(eos)[0].strip()

    sql = ""
    expl = ""

    # ---------- 1) Labeled "SQL:" format ----------
    # Stop SQL at Explanation or any [ALLCAPS] tag like [ANSWER], [NOTES], etc.
    sql_match = re.search(
        r"(?:^|\n)\s*SQL\s*:\s*(.*?)(?=\n(?:Explanation|\[EXPLANATION\])\s*:|\n\[[A-Z ]+\]|$)",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    if sql_match:
        sql = sql_match.group(1).strip()

    # Explanation: after "Explanation:" or "[EXPLANATION]" until next [TAG] or end
    exp_match = re.search(
        r"(?:Explanation|\[EXPLANATION\])\s*:\s*(.*?)(?=\n\[[A-Z ]+\]|$)",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    if exp_match:
        expl = exp_match.group(1).strip()

    # ---------- 2) Fallbacks for base style ----------
    # If SQL still empty, grab first SELECT ...;
    if not sql:
        sel_match = re.search(r"(?is)\bselect\b.*?;", text)
        if sel_match:
            sql = sel_match.group(0).strip()

    # If explanation still empty, use [EXPLANATION] block if present
    if not expl:
        ex_block = re.search(
            r"\[EXPLANATION\]\s*(.*?)(?=\n\[[A-Z ]+\]|$)",
            text,
            flags=re.DOTALL,
        )
        if ex_block:
            expl = ex_block.group(1).strip()

    # ---------- 3) Clean SQL: strip any leftover section tags ----------
    for marker in ["[ANSWER]", "[EXPLANATION]", "[NOTES]", "[REASONING]", "[STEPS]"]:
        if marker in sql:
            sql = sql.split(marker)[0].strip()

    # ---------- 4) Last fallback ----------
    if not expl and sql:
        expl = "(No explanation parsed.)"

    return sql, expl

def exact_match(pred: str, ref: str) -> bool:
    return normalize_sql(pred) == normalize_sql(ref)

print("✅ Helpers ready (EOS guess:", repr(get_eos()), ")")


✅ Helpers ready (EOS guess: '<|im_end|>' )


## 4) Load Gretel dataset from Hugging Face (**ONLINE-ONLY**)
This notebook strictly downloads `gretelai/synthetic_text_to_sql`. No local files or fallback are used.

In [4]:

from datasets import load_dataset, DatasetDict

def map_gretel_fields(example):
    # Gretel -> our format
    return {
        "schema": example.get("sql_context", ""),
        "question": example.get("sql_prompt", ""),
        "sql": example.get("sql", ""),
        "explanation": example.get("sql_explanation", ""),
    }

print("Downloading gretelai/synthetic_text_to_sql from Hugging Face...")
raw = load_dataset("gretelai/synthetic_text_to_sql")  # requires internet

# Map fields for train/test
train_mapped = raw["train"].map(map_gretel_fields, remove_columns=raw["train"].column_names)
test_mapped  = raw["test"].map(map_gretel_fields,  remove_columns=raw["test"].column_names)

print("Creating smaller dataset subsets...")
train_subset = train_mapped.select(range(1000))      # 40,000 training
val_subset = train_mapped.select(range(500, 1000)) # 10,000 validation (next 10k)
test_subset = test_mapped.select(range(400))         # 3,000 test

ds = DatasetDict({
    "train": train_subset,
    "validation": val_subset,
    "test": test_subset
})

print(ds)
print("✅ Loaded Gretel dataset ONLINE from Hugging Face.")
print(f"✅ Using custom splits:")
print(f"   Train: {len(ds['train'])} examples")
print(f"   Validation: {len(ds['validation'])} examples")
print(f"   Test: {len(ds['test'])} examples")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5851 [00:00<?, ? examples/s]

Creating smaller dataset subsets...
DatasetDict({
    train: Dataset({
        features: ['sql', 'schema', 'question', 'explanation'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['sql', 'schema', 'question', 'explanation'],
        num_rows: 500
    })
    test: Dataset({
        features: ['sql', 'schema', 'question', 'explanation'],
        num_rows: 400
    })
})
✅ Loaded Gretel dataset ONLINE from Hugging Face.
✅ Using custom splits:
   Train: 1000 examples
   Validation: 500 examples
   Test: 400 examples


## 5) Baseline Evaluation (pre-fine-tune)

In [20]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import sacrebleu
import pandas as pd
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
model_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

gen_cfg = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS,
    min_new_tokens=MIN_NEW_TOKENS,
    num_beams=BEAM_SIZE,
    do_sample=False,
    repetition_penalty=REPETITION_PENALTY,
    eos_token_id=tokenizer.eos_token_id,
)

def generate_response(model, schema: str, question: str) -> str:
    prompt = build_prompt(schema, question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, generation_config=gen_cfg)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

def evaluate_split(model, split_name: str, max_eval: int = 100) -> pd.DataFrame:
    assert split_name in ds, f"Split {split_name} not found"
    rows = []
    dataset = ds[split_name]
    n = len(dataset) if max_eval is None else min(len(dataset), max_eval)
    for i in range(n):
        ex = dataset[i]
        pred_text = generate_response(model, ex["schema"], ex["question"])
        pred_sql, pred_expl = split_response(pred_text)
        rows.append({
            "schema": ex["schema"],
            "question": ex["question"],
            "ref_sql": ex["sql"],
            "ref_explanation": ex.get("explanation", ""),
            "pred_sql": pred_sql,
            "pred_explanation": pred_expl,
        })
    return pd.DataFrame(rows)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
baseline_df = evaluate_split(model_base, "test", max_eval=100)
baseline_csv = f"{OUTPUT_DIR}/baseline_predictions.csv"
baseline_df.to_csv(baseline_csv, index=False)
print(f"💾 Saved baseline predictions to: {baseline_csv}")

def compute_metrics(df: pd.DataFrame) -> dict:
    em = (df.apply(lambda r: exact_match(r["pred_sql"], r["ref_sql"]), axis=1)).mean()
    preds = [normalize_sql(s) for s in df["pred_sql"].tolist()]
    refs  = [[normalize_sql(s) for s in df["ref_sql"].tolist()]]
    bleu = sacrebleu.corpus_bleu(preds, refs).score / 100.0
    syntax_fail = df["pred_sql"].apply(lambda s: 0 if any(tok in s.lower() for tok in ["select", "from"]) else 1).mean()
    return {"exact_match": em, "bleu": bleu, "syntax_fail_rate": syntax_fail}

baseline_metrics = compute_metrics(baseline_df)
baseline_metrics


💾 Saved baseline predictions to: ./outputs/baseline_predictions.csv


{'exact_match': np.float64(0.08),
 'bleu': 0.27005855307512017,
 'syntax_fail_rate': np.float64(0.45)}

## 6) LoRA Fine-Tuning

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

def format_example(ex: Dict[str, Any]) -> Dict[str, Any]:
    prompt = build_prompt(ex["schema"], ex["question"])
    target = build_target(ex["sql"], ex.get("explanation", ""))
    return {"text": prompt + target}

train_ds_fmt = ds["train"].map(format_example, remove_columns=ds["train"].column_names)
val_ds_fmt   = ds["validation"].map(format_example, remove_columns=ds["validation"].column_names)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_tok = train_ds_fmt.map(tokenize_fn, batched=True, remove_columns=train_ds_fmt.column_names)
val_tok   = val_ds_fmt.map(tokenize_fn, batched=True, remove_columns=val_ds_fmt.column_names)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    task_type=TaskType.CAUSAL_LM
)

model_tune = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
model_tune = get_peft_model(model_tune, peft_config)
model_tune.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LR,
    weight_decay=0.0,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=max(100, LOGGING_STEPS),
    save_strategy="steps",
    save_steps=max(100, LOGGING_STEPS),
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to=[],
)

trainer = Trainer(
    model=model_tune,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

train_result = trainer.train()
print(train_result)

ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapters"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"💾 LoRA adapters saved to: {ADAPTER_DIR}")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
100,1.019500,0.889126
200,0.788600,0.745877
300,0.767500,0.723380
400,0.730000,0.710378
500,0.687400,0.702864
600,0.724800,0.698626
700,0.706000,0.696680
800,0.721200,0.694699
900,0.712100,0.693963
1000,0.690700,0.693925


TrainOutput(global_step=1000, training_loss=0.7781774635314942, metrics={'train_runtime': 983.6062, 'train_samples_per_second': 1.017, 'train_steps_per_second': 1.017, 'total_flos': 437217176825088.0, 'train_loss': 0.7781774635314942, 'epoch': 1.0})
💾 LoRA adapters saved to: ./outputs/lora_adapters


## 7) Post-tune Evaluation & 8) Comparison

In [ ]:
from peft import PeftModel
import pandas as pd
from transformers import AutoModelForCausalLM # Added import

model_ft = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
model_ft = PeftModel.from_pretrained(model_ft, ADAPTER_DIR)

def evaluate_split_with_model(model, split_name: str, max_eval: int = 100) -> pd.DataFrame:
    rows = []
    dataset = ds[split_name]
    n = len(dataset) if max_eval is None else min(len(dataset), max_eval)
    for i in range(n):
        ex = dataset[i]
        pred_text = generate_response(model, ex["schema"], ex["question"])
        pred_sql, pred_expl = split_response(pred_text)
        rows.append({
            "schema": ex["schema"],
            "question": ex["question"],
            "ref_sql": ex["sql"],
            "ref_explanation": ex.get("explanation", ""),
            "pred_sql": pred_sql,
            "pred_explanation": pred_expl,
        })
    return pd.DataFrame(rows)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
ft_csv = f"{OUTPUT_DIR}/finetuned_predictions.csv"
ft_df = evaluate_split_with_model(model_ft, "test", max_eval=100) # Added this line to create ft_df
ft_df.to_csv(ft_csv, index=False)
print(f"💾 Saved finetuned predictions to: {ft_csv}")

def compute_metrics(df: pd.DataFrame) -> dict:
    em = (df.apply(lambda r: exact_match(r["pred_sql"], r["ref_sql"]), axis=1)).mean()
    preds = [normalize_sql(s) for s in df["pred_sql"].tolist()]
    refs  = [[normalize_sql(s) for s in df["ref_sql"].tolist()]]
    import sacrebleu
    bleu = sacrebleu.corpus_bleu(preds, refs).score / 100.0
    syntax_fail = df["pred_sql"].apply(lambda s: 0 if any(tok in s.lower() for tok in ["select", "from"]) else 1).mean()
    return {"exact_match": em, "bleu": bleu, "syntax_fail_rate": syntax_fail}

import pandas as pd
baseline_df = pd.read_csv(f"{OUTPUT_DIR}/baseline_predictions.csv")

baseline_metrics = compute_metrics(baseline_df)
ft_metrics = compute_metrics(ft_df)

summary = pd.DataFrame([
    {"model": "base", **baseline_metrics},
    {"model": "finetuned", **ft_metrics}
]).set_index("model")
display(summary)

# Examples where FT improved EM
merged = baseline_df[["schema","question","ref_sql","pred_sql"]].rename(columns={"pred_sql":"base_sql"}).merge(
    ft_df[["question","pred_sql"]].rename(columns={"pred_sql":"ft_sql"}),
    on="question",
    how="inner"
)
def em_flag(a,b):
    return normalize_sql(a)==normalize_sql(b)

merged["base_em"] = merged.apply(lambda r: em_flag(r["base_sql"], r["ref_sql"]), axis=1)
merged["ft_em"]   = merged.apply(lambda r: em_flag(r["ft_sql"], r["ref_sql"]), axis=1)

improved = merged[(merged["base_em"]==False) & (merged["ft_em"]==True)].head(5)
display(improved)

💾 Saved finetuned predictions to: ./outputs/finetuned_predictions.csv


,exact_match,bleu,syntax_fail_rate
model,,,
base,0.0,0.107418,0.12
finetuned,0.2,0.492131,0.06


,schema,question,ref_sql,base_sql,ft_sql,base_em,ft_em
4,"CREATE TABLE Movies_Release_Year (id INT, titl...",What is the total budget for movies released b...,SELECT SUM(budget) FROM Movies_Release_Year WH...,SELECT SUM(budget) FROM Movies_Release_Year WH...,SELECT SUM(budget) FROM Movies_Release_Year WH...,False,True
5,"CREATE TABLE attorneys (attorney_id INT, attor...",Add a new attorney named 'Oliver Martinez' wit...,"INSERT INTO attorneys (attorney_name, attorney...","```sql\nINSERT INTO attorneys (attorney_id, at...","INSERT INTO attorneys (attorney_name, attorney...",False,True
8,"CREATE TABLE marine_species (name TEXT, conser...",List all marine species with their conservatio...,"SELECT name, conservation_status FROM marine_s...","SELECT name, conservation_status FROM marine_s...","SELECT name, conservation_status FROM marine_s...",False,True
12,"CREATE TABLE emergency_calls (id INT, city VAR...",What is the maximum response time for emergenc...,SELECT MAX(response_time) FROM emergency_calls...,15\n\n[EXPLANATION]\nThe maximum response time...,SELECT MAX(response_time) FROM emergency_calls...,False,True
15,"CREATE TABLE arms_imports (id INT PRIMARY KEY,...",Delete arms_imports table records where year i...,DELETE FROM arms_imports WHERE year < 2000;,DELETE FROM `arms_imports` WHERE `year` < 2000...,DELETE FROM arms_imports WHERE year < 2000;,False,True


## 9) Gradio UI — Base vs Fine-tuned

In [21]:

import gradio as gr

def infer(schema: str, question: str, model_choice: str) -> tuple:
    model = model_ft if model_choice == "Fine-tuned" else model_base
    text = generate_response(model, schema, question)
    sql, expl = split_response(text)
    print("\n===== DEBUG =====")
    print("RAW OUTPUT:\n", text)
    print("PARSED SQL:\n", sql)
    print("PARSED EXPLANATION:\n", expl)
    print("=================\n")
    if not sql.strip():
        sql = "(No SQL parsed — check schema & question.)"
    if not expl.strip():
        expl = "(No explanation parsed.)"
    return sql, expl

with gr.Blocks() as demo:
    gr.Markdown("# Text-to-SQL Demo — Base vs Fine-tuned (Online Gretel)")
    with gr.Row():
        with gr.Column():
            schema = gr.Textbox(label="Schema (DDL or columns)", lines=12, placeholder="CREATE TABLE ...")
            question = gr.Textbox(label="Natural Language Question", lines=3, placeholder="e.g., List orders in 2024 by city")
            model_choice = gr.Radio(choices=["Base", "Fine-tuned"], value="Fine-tuned", label="Model")
            btn = gr.Button("Generate")
        with gr.Column():
            sql_out = gr.Textbox(label="SQL", lines=8)
            exp_out = gr.Textbox(label="Explanation", lines=6)
    btn.click(infer, inputs=[schema, question, model_choice], outputs=[sql_out, exp_out])

print("✅ UI is defined. Call demo.launch() to start locally.")
demo.launch(share=False)


✅ UI is defined. Call demo.launch() to start locally.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [19]:
# 1) Simulate BASE model style (with [ANSWER] and [EXPLANATION])

sample_base = """<s>[SCHEMA]
CREATE TABLE Customer (
    cust_id INT PRIMARY KEY,
    cust_name VARCHAR(50),
    city VARCHAR(30),
    phone VARCHAR(15)
);

[QUESTION]
Display all customers from the city 'Boston'

[RESPONSE_FORMAT]
SQL:  SELECT cust_name FROM Customer WHERE city = 'Boston' LIMIT 10;

[ANSWER]
SELECT cust_name FROM Customer WHERE city = 'Boston' LIMIT 10;
[EXPLANATION]
This SQL query selects all customers from the city 'Boston' and displays them in a single row with a limit of 10 rows.
"""

print("BASE STYLE →", split_response(sample_base))


# 2) Simulate FINE-TUNED style (our training format)

sample_ft = """<s>[SCHEMA]
CREATE TABLE orders (id INT, amount INT);

[QUESTION]
Show all orders above 100.

[RESPONSE_FORMAT]
SQL: SQL: SELECT * FROM orders WHERE amount > 100;
Explanation: This query returns all orders where amount is greater than 100.<|im_end|>"""

print("FINE-TUNED STYLE →", split_response(sample_ft))


BASE STYLE → ("SELECT cust_name FROM Customer WHERE city = 'Boston' LIMIT 10;", "This SQL query selects all customers from the city 'Boston' and displays them in a single row with a limit of 10 rows.")
FINE-TUNED STYLE → ('SQL: SELECT * FROM orders WHERE amount > 100;', 'This query returns all orders where amount is greater than 100.')
